# Prepare Future Climate Data

This notebook prepares the CNRM-CM6-1 climate data that will be used for future glacier mass balance predictions. The historical CNRM data will first be compared with ERA5 so differences between the two climate datasets can be corrected before the future SSP2-4.5 projections are used.

The future data will then be processed using the same hydrological year and seasonal definitions used for the historical model data. The final goal is to produce the same four climate features used by the selected Linear Regression model.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import xarray as xr

DATA_DIR = Path("..") / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

future_temperature_file = (
    RAW_DIR /
    "tas_Amon_CNRM-CM6-1_ssp245_r1i1p1f2_gr_20260116-20761216.nc"
)

future_precipitation_file = (
    RAW_DIR /
    "pr_Amon_CNRM-CM6-1_ssp245_r1i1p1f2_gr_20260116-20761216.nc"
)

print("Future temperature exists:", future_temperature_file.exists())
print("Future precipitation exists:", future_precipitation_file.exists())

Future temperature exists: True
Future precipitation exists: True


In [5]:
future_temperature = xr.open_dataset(future_temperature_file)
future_precipitation = xr.open_dataset(future_precipitation_file)

print("Future Temperature Dataset")
print(future_temperature)

print("Future Precipitation Dataset")
print(future_precipitation)

Future Temperature Dataset
<xarray.Dataset> Size: 80MB
Dimensions:      (time: 612, axis_nbounds: 2, lat: 128, lon: 256)
Coordinates:
  * time         (time) datetime64[ns] 5kB 2026-01-16T12:00:00 ... 2076-12-16...
  * lat          (lat) float64 1kB -88.93 -87.54 -86.14 ... 86.14 87.54 88.93
  * lon          (lon) float64 2kB 0.0 1.406 2.812 4.219 ... 355.8 357.2 358.6
    height       float64 8B ...
Dimensions without coordinates: axis_nbounds
Data variables:
    time_bounds  (time, axis_nbounds) datetime64[ns] 10kB ...
    tas          (time, lat, lon) float32 80MB ...
Attributes: (12/52)
    Conventions:            CF-1.7 CMIP-6.2
    creation_date:          2018-11-15T10:39:09Z
    description:            Future scenario with medium radiative forcing by ...
    title:                  CNRM-CM6-1 model output prepared for CMIP6 / Scen...
    activity_id:            ScenarioMIP
    contact:                contact.cmip@meteo.fr
    ...                     ...
    dr2xml_md5sum:       

In [6]:
print("Temperature")
print("Variable:", "tas")
print("Shape:", future_temperature["tas"].shape)
print("Units:", future_temperature["tas"].attrs.get("units"))
print(
    "Time:",
    future_temperature["time"].values[0],
    "to",
    future_temperature["time"].values[-1]
)

print("\nPrecipitation")
print("Variable:", "pr")
print("Shape:", future_precipitation["pr"].shape)
print("Units:", future_precipitation["pr"].attrs.get("units"))
print(
    "Time:",
    future_precipitation["time"].values[0],
    "to",
    future_precipitation["time"].values[-1]
)

Temperature
Variable: tas
Shape: (612, 128, 256)
Units: K
Time: 2026-01-16T12:00:00.000000000 to 2076-12-16T12:00:00.000000000

Precipitation
Variable: pr
Shape: (612, 128, 256)
Units: kg m-2 s-1
Time: 2026-01-16T12:00:00.000000000 to 2076-12-16T12:00:00.000000000


### Check Future Climate Coordinates

The temperature and precipitation files should use the same monthly time steps and spatial grid. Their coordinates are checked before any conversions or spatial processing are performed.

In [7]:
print(
    "Matching time:",
    np.array_equal(
        future_temperature["time"].values,
        future_precipitation["time"].values
    )
)

print(
    "Matching latitude:",
    np.array_equal(
        future_temperature["lat"].values,
        future_precipitation["lat"].values
    )
)

print(
    "Matching longitude:",
    np.array_equal(
        future_temperature["lon"].values,
        future_precipitation["lon"].values
    )
)

print("\nLatitude range:")
print(
    future_temperature["lat"].values.min(),
    "to",
    future_temperature["lat"].values.max()
)

print("\nLongitude range:")
print(
    future_temperature["lon"].values.min(),
    "to",
    future_temperature["lon"].values.max()
)

Matching time: True
Matching latitude: True
Matching longitude: True

Latitude range:
-88.92773535220698 to 88.92773535220698

Longitude range:
0.0 to 358.59375


### Locate Mont Blanc on the CNRM Grid

The CNRM-CM6-1 climate grid is much coarser than the Mont Blanc study area. Before extracting climate values, the nearest latitude and longitude grid points are checked to see how the study area falls within the model grid.

In [8]:
mont_blanc_lat = (45.783327 + 45.992668) / 2
mont_blanc_lon = (6.674194 + 7.043610) / 2

nearest_lat_index = np.abs(
    future_temperature["lat"].values - mont_blanc_lat
).argmin()

nearest_lon_index = np.abs(
    future_temperature["lon"].values - mont_blanc_lon
).argmin()

print(f"Mont Blanc center: {mont_blanc_lat:.4f}, {mont_blanc_lon:.4f}")

print("\nNearby latitude grid points:")
print(
    future_temperature["lat"].values[
        nearest_lat_index - 2:nearest_lat_index + 3
    ]
)

print("\nNearby longitude grid points:")
print(
    future_temperature["lon"].values[
        nearest_lon_index - 2:nearest_lon_index + 3
    ]
)

print("\nNearest CNRM grid point:")
print(
    float(future_temperature["lat"].values[nearest_lat_index]),
    float(future_temperature["lon"].values[nearest_lon_index])
)

Mont Blanc center: 45.8880, 6.8589

Nearby latitude grid points:
[42.72333486 44.12409297 45.5248501  46.92560615 48.32636102]

Nearby longitude grid points:
[4.21875 5.625   7.03125 8.4375  9.84375]

Nearest CNRM grid point:
45.52485010125677 7.03125


### Load Mont Blanc Glacier Locations

The historical model dataset contains the locations of the Mont Blanc glacier records used during model training. These same locations will be used to extract future CNRM climate values so the future inputs remain spatially consistent with the historical model data.

In [9]:
historical_model_file = (
    PROCESSED_DIR /
    "MontBlanc_Historical_Model_Data_1968-2015.csv"
)

historical_data = pd.read_csv(historical_model_file)

glacier_locations = (
    historical_data[
        ["glacier_index", "longitude", "latitude"]
    ]
    .drop_duplicates()
    .sort_values("glacier_index")
    .reset_index(drop=True)
)

print("Historical model data exists:", historical_model_file.exists())
print("Number of glacier records:", len(glacier_locations))

print("\nLongitude range:")
print(
    glacier_locations["longitude"].min(),
    "to",
    glacier_locations["longitude"].max()
)

print("\nLatitude range:")
print(
    glacier_locations["latitude"].min(),
    "to",
    glacier_locations["latitude"].max()
)

glacier_locations.head()

Historical model data exists: True
Number of glacier records: 58

Longitude range:
6.776 to 7.008

Latitude range:
45.784 to 45.992


,glacier_index,longitude,latitude
0,423,6.776,45.799
1,424,6.778,45.809
2,427,6.783,45.817
3,429,6.784,45.784
4,431,6.785,45.822


### Interpolate Future Climate to Glacier Locations

The CNRM-CM6-1 grid is much larger than the individual Mont Blanc glaciers, so using only the nearest climate grid point would assign the same climate values to many glacier records. Bilinear interpolation is used to estimate temperature and precipitation at each glacier location from the surrounding CNRM grid points. This keeps the future climate inputs spatially consistent with the glacier locations used in the historical model dataset.

In [10]:
glacier_latitudes = xr.DataArray(
    glacier_locations["latitude"].values,
    dims="glacier",
    coords={"glacier": glacier_locations["glacier_index"].values}
)

glacier_longitudes = xr.DataArray(
    glacier_locations["longitude"].values,
    dims="glacier",
    coords={"glacier": glacier_locations["glacier_index"].values}
)

future_temperature_glaciers = future_temperature["tas"].interp(
    lat=glacier_latitudes,
    lon=glacier_longitudes
)

future_precipitation_glaciers = future_precipitation["pr"].interp(
    lat=glacier_latitudes,
    lon=glacier_longitudes
)

print("Temperature shape:", future_temperature_glaciers.shape)
print("Precipitation shape:", future_precipitation_glaciers.shape)

print("\nTemperature missing values:",
      int(future_temperature_glaciers.isnull().sum()))

print("Precipitation missing values:",
      int(future_precipitation_glaciers.isnull().sum()))

print("\nGlacier records:",
      future_temperature_glaciers.sizes["glacier"])

print("Monthly time steps:",
      future_temperature_glaciers.sizes["time"])

Temperature shape: (612, 58)
Precipitation shape: (612, 58)

Temperature missing values: 0
Precipitation missing values: 0

Glacier records: 58
Monthly time steps: 612


### Convert Future Climate Units

The future CNRM temperature and precipitation variables use different units from the historical model features. Temperature is converted from Kelvin to degrees Celsius. Precipitation is provided as a mean flux in kilograms per square meter per second, which is equivalent to millimeters of water per second, so it is converted to a monthly precipitation total using the number of days in each month.

In [11]:
future_temperature_c = future_temperature_glaciers - 273.15

days_in_month = future_precipitation_glaciers["time"].dt.days_in_month
seconds_in_month = days_in_month * 24 * 60 * 60

future_precipitation_mm = (
    future_precipitation_glaciers * seconds_in_month
)

print("Temperature units: °C")
print(
    "Temperature range:",
    float(future_temperature_c.min()),
    "to",
    float(future_temperature_c.max())
)

print("\nPrecipitation units: mm/month")
print(
    "Precipitation range:",
    float(future_precipitation_mm.min()),
    "to",
    float(future_precipitation_mm.max())
)

print("\nTemperature missing values:",
      int(future_temperature_c.isnull().sum()))

print("Precipitation missing values:",
      int(future_precipitation_mm.isnull().sum()))

Temperature units: °C
Temperature range: -7.575658631369379 to 18.870709415950273

Precipitation units: mm/month
Precipitation range: 9.24171014252931 to 423.6263898919689

Temperature missing values: 0
Precipitation missing values: 0


### Assign Hydrological Years

The historical model uses an October through September hydrological year rather than a calendar year. The future climate data are assigned to the same hydrological-year structure so the future seasonal features match those used during model training.

Because the future CNRM data begin in January 2026 and end in December 2076, the first and last hydrological years are incomplete. Only complete hydrological years will be retained.

In [12]:
future_hydrological_year = xr.where(
    future_temperature_c["time"].dt.month >= 10,
    future_temperature_c["time"].dt.year + 1,
    future_temperature_c["time"].dt.year
)

future_temperature_c = future_temperature_c.assign_coords(
    hydrological_year=("time", future_hydrological_year.data)
)

future_precipitation_mm = future_precipitation_mm.assign_coords(
    hydrological_year=("time", future_hydrological_year.data)
)

months_per_hydrological_year = (
    future_temperature_c["time"]
    .groupby("hydrological_year")
    .count()
)

print(months_per_hydrological_year.to_series())

hydrological_year
2026     9
2027    12
2028    12
2029    12
2030    12
2031    12
2032    12
2033    12
2034    12
2035    12
2036    12
2037    12
2038    12
2039    12
2040    12
2041    12
2042    12
2043    12
2044    12
2045    12
2046    12
2047    12
2048    12
2049    12
2050    12
2051    12
2052    12
2053    12
2054    12
2055    12
2056    12
2057    12
2058    12
2059    12
2060    12
2061    12
2062    12
2063    12
2064    12
2065    12
2066    12
2067    12
2068    12
2069    12
2070    12
2071    12
2072    12
2073    12
2074    12
2075    12
2076    12
2077     3
Name: time, dtype: int64


### Keep Complete Future Hydrological Years

Only hydrological years containing all 12 months are kept for the future model inputs. This removes the incomplete 2026 and 2077 hydrological years and leaves 50 complete years from 2027 through 2076.

In [13]:
complete_future_years = (
    months_per_hydrological_year
    .where(months_per_hydrological_year == 12, drop=True)
    ["hydrological_year"]
    .values
)

future_temperature_complete = future_temperature_c.where(
    future_temperature_c["hydrological_year"].isin(complete_future_years),
    drop=True
)

future_precipitation_complete = future_precipitation_mm.where(
    future_precipitation_mm["hydrological_year"].isin(complete_future_years),
    drop=True
)

print("First complete hydrological year:",
      int(complete_future_years.min()))

print("Last complete hydrological year:",
      int(complete_future_years.max()))

print("Number of complete hydrological years:",
      len(complete_future_years))

print("\nTemperature shape:",
      future_temperature_complete.shape)

print("Precipitation shape:",
      future_precipitation_complete.shape)

First complete hydrological year: 2027
Last complete hydrological year: 2076
Number of complete hydrological years: 50

Temperature shape: (600, 58)
Precipitation shape: (600, 58)


### Create Future Seasonal Climate Features

The final model uses four seasonal climate variables: winter temperature, summer temperature, winter precipitation, and summer precipitation. The future CNRM data are grouped using the same seasons as the historical model, with winter defined as October through March and summer defined as April through September.

Seasonal temperature is calculated as the mean monthly temperature, while seasonal precipitation is calculated as the total precipitation across the six month season.

In [14]:
winter_months = [10, 11, 12, 1, 2, 3]
summer_months = [4, 5, 6, 7, 8, 9]

future_winter_temperature = (
    future_temperature_complete
    .where(
        future_temperature_complete["time"].dt.month.isin(winter_months),
        drop=True
    )
    .groupby("hydrological_year")
    .mean(dim="time")
)

future_summer_temperature = (
    future_temperature_complete
    .where(
        future_temperature_complete["time"].dt.month.isin(summer_months),
        drop=True
    )
    .groupby("hydrological_year")
    .mean(dim="time")
)

future_winter_precipitation = (
    future_precipitation_complete
    .where(
        future_precipitation_complete["time"].dt.month.isin(winter_months),
        drop=True
    )
    .groupby("hydrological_year")
    .sum(dim="time")
)

future_summer_precipitation = (
    future_precipitation_complete
    .where(
        future_precipitation_complete["time"].dt.month.isin(summer_months),
        drop=True
    )
    .groupby("hydrological_year")
    .sum(dim="time")
)

print("Winter temperature shape:",
      future_winter_temperature.shape)

print("Summer temperature shape:",
      future_summer_temperature.shape)

print("Winter precipitation shape:",
      future_winter_precipitation.shape)

print("Summer precipitation shape:",
      future_summer_precipitation.shape)

print("\nHydrological years:",
      int(future_winter_temperature["hydrological_year"].min()),
      "to",
      int(future_winter_temperature["hydrological_year"].max()))

Winter temperature shape: (50, 58)
Summer temperature shape: (50, 58)
Winter precipitation shape: (50, 58)
Summer precipitation shape: (50, 58)

Hydrological years: 2027 to 2076


### Check Future Seasonal Climate Values

The seasonal climate features are checked before they are combined into the final future dataset. Their ranges should contain reasonable temperature and precipitation values with no missing data.

In [15]:
seasonal_features = {
    "Winter Temperature (°C)": future_winter_temperature,
    "Summer Temperature (°C)": future_summer_temperature,
    "Winter Precipitation (mm)": future_winter_precipitation,
    "Summer Precipitation (mm)": future_summer_precipitation
}

for name, values in seasonal_features.items():
    print(name)
    print("  Minimum:", float(values.min()))
    print("  Maximum:", float(values.max()))
    print("  Missing values:", int(values.isnull().sum()))
    print()

Winter Temperature (°C)
  Minimum: -0.02015027952848906
  Maximum: 4.432925101231102
  Missing values: 0

Summer Temperature (°C)
  Minimum: 11.239559839393118
  Maximum: 13.994665723952911
  Missing values: 0

Winter Precipitation (mm)
  Minimum: 654.6016805012969
  Maximum: 1295.6049717909586
  Missing values: 0

Summer Precipitation (mm)
  Minimum: 834.7797923355149
  Maximum: 1580.2586454346215
  Missing values: 0



### Combine Future Seasonal Climate Features

The four seasonal climate variables are combined into one row for each glacier and hydrological year. This creates the same climate feature structure used by the historical model while keeping the CNRM values uncorrected for now. Bias correction will be applied before the future dataset is used for mass balance prediction.

In [16]:
future_climate = pd.DataFrame({
    "hydrological_year": np.repeat(
        future_winter_temperature["hydrological_year"].values,
        len(glacier_locations)
    ),
    "glacier_index": np.tile(
        glacier_locations["glacier_index"].values,
        len(complete_future_years)
    ),
    "winter_temperature_c": future_winter_temperature.values.flatten(),
    "summer_temperature_c": future_summer_temperature.values.flatten(),
    "winter_precipitation_mm": future_winter_precipitation.values.flatten(),
    "summer_precipitation_mm": future_summer_precipitation.values.flatten()
})

future_climate = future_climate.merge(
    glacier_locations,
    on="glacier_index",
    how="left"
)

print("Future climate shape:", future_climate.shape)

print(
    "Hydrological years:",
    future_climate["hydrological_year"].min(),
    "to",
    future_climate["hydrological_year"].max()
)

print("Glacier records:", future_climate["glacier_index"].nunique())

print(
    "Duplicate glacier-year rows:",
    future_climate.duplicated(
        subset=["glacier_index", "hydrological_year"]
    ).sum()
)

print("\nMissing values:")
print(future_climate.isnull().sum())

future_climate.head()

Future climate shape: (2900, 8)
Hydrological years: 2027 to 2076
Glacier records: 58
Duplicate glacier-year rows: 0

Missing values:
hydrological_year          0
glacier_index              0
winter_temperature_c       0
summer_temperature_c       0
winter_precipitation_mm    0
summer_precipitation_mm    0
longitude                  0
latitude                   0
dtype: int64


,hydrological_year,glacier_index,winter_temperature_c,summer_temperature_c,winter_precipitation_mm,summer_precipitation_mm,longitude,latitude
0,2027,423,2.684208,12.168462,907.694857,1060.957402,6.776,45.799
1,2027,424,2.694755,12.181449,908.158735,1060.038048,6.778,45.809
2,2027,427,2.698998,12.187856,908.780980,1059.842492,6.783,45.817
3,2027,429,2.654991,12.136423,907.734716,1064.026330,6.784,45.784
4,2027,431,2.703016,12.193140,909.095761,1059.548266,6.785,45.822


### Load Historical CNRM Climate Data

Historical CNRM-CM6-1 temperature and precipitation data are loaded so they can be compared with the ERA5 climate data used to build the historical model dataset. The overlapping historical period will be used to measure the systematic difference between CNRM and ERA5 before the correction is applied to the future climate features.

In [18]:
historical_temperature_file = (
    RAW_DIR /
    "tas_Amon_CNRM-CM6-1_historical_r1i1p1f2_gr_19670116-20141216.nc"
)

historical_precipitation_file = (
    RAW_DIR /
    "pr_Amon_CNRM-CM6-1_historical_r1i1p1f2_gr_19670116-20141216.nc"
)

historical_temperature = xr.open_dataset(historical_temperature_file)
historical_precipitation = xr.open_dataset(historical_precipitation_file)

print("Historical temperature exists:",
      historical_temperature_file.exists())

print("Historical precipitation exists:",
      historical_precipitation_file.exists())

print("\nTemperature")
print("Variable:", "tas")
print("Shape:", historical_temperature["tas"].shape)
print("Units:", historical_temperature["tas"].attrs.get("units"))
print(
    "Time:",
    historical_temperature["time"].values[0],
    "to",
    historical_temperature["time"].values[-1]
)

print("\nPrecipitation")
print("Variable:", "pr")
print("Shape:", historical_precipitation["pr"].shape)
print("Units:", historical_precipitation["pr"].attrs.get("units"))
print(
    "Time:",
    historical_precipitation["time"].values[0],
    "to",
    historical_precipitation["time"].values[-1]
)

Historical temperature exists: True
Historical precipitation exists: True

Temperature
Variable: tas
Shape: (576, 128, 256)
Units: K
Time: 1967-01-16T12:00:00.000000000 to 2014-12-16T12:00:00.000000000

Precipitation
Variable: pr
Shape: (576, 128, 256)
Units: kg m-2 s-1
Time: 1967-01-16T12:00:00.000000000 to 2014-12-16T12:00:00.000000000


### Prepare Historical CNRM Climate Data

The historical CNRM temperature and precipitation data are processed using the same glacier locations and unit conversions as the future CNRM data. Keeping the processing consistent allows the historical CNRM climate to be compared directly with the ERA5 climate features used during model development.

In [19]:
historical_temperature_glaciers = historical_temperature["tas"].interp(
    lat=glacier_latitudes,
    lon=glacier_longitudes
)

historical_precipitation_glaciers = historical_precipitation["pr"].interp(
    lat=glacier_latitudes,
    lon=glacier_longitudes
)

historical_temperature_c = historical_temperature_glaciers - 273.15

historical_days_in_month = (
    historical_precipitation_glaciers["time"].dt.days_in_month
)

historical_seconds_in_month = (
    historical_days_in_month * 24 * 60 * 60
)

historical_precipitation_mm = (
    historical_precipitation_glaciers * historical_seconds_in_month
)

print("Temperature shape:", historical_temperature_c.shape)
print("Precipitation shape:", historical_precipitation_mm.shape)

print("\nTemperature range:",
      float(historical_temperature_c.min()),
      "to",
      float(historical_temperature_c.max()))

print("Precipitation range:",
      float(historical_precipitation_mm.min()),
      "to",
      float(historical_precipitation_mm.max()))

print("\nTemperature missing values:",
      int(historical_temperature_c.isnull().sum()))

print("Precipitation missing values:",
      int(historical_precipitation_mm.isnull().sum()))

Temperature shape: (576, 58)
Precipitation shape: (576, 58)

Temperature range: -8.632648928045171 to 17.529385864611868
Precipitation range: 5.18826105247078 to 399.88247017684427

Temperature missing values: 0
Precipitation missing values: 0


### Assign Historical CNRM Hydrological Years

The historical CNRM data are assigned to the same October through September hydrological year used throughout the project. The first and last hydrological years are checked for completeness before seasonal climate features are created.

In [20]:
historical_hydrological_year = xr.where(
    historical_temperature_c["time"].dt.month >= 10,
    historical_temperature_c["time"].dt.year + 1,
    historical_temperature_c["time"].dt.year
)

historical_temperature_c = historical_temperature_c.assign_coords(
    hydrological_year=("time", historical_hydrological_year.data)
)

historical_precipitation_mm = historical_precipitation_mm.assign_coords(
    hydrological_year=("time", historical_hydrological_year.data)
)

historical_months_per_year = (
    historical_temperature_c["time"]
    .groupby("hydrological_year")
    .count()
)

print(historical_months_per_year.to_series())

hydrological_year
1967     9
1968    12
1969    12
1970    12
1971    12
1972    12
1973    12
1974    12
1975    12
1976    12
1977    12
1978    12
1979    12
1980    12
1981    12
1982    12
1983    12
1984    12
1985    12
1986    12
1987    12
1988    12
1989    12
1990    12
1991    12
1992    12
1993    12
1994    12
1995    12
1996    12
1997    12
1998    12
1999    12
2000    12
2001    12
2002    12
2003    12
2004    12
2005    12
2006    12
2007    12
2008    12
2009    12
2010    12
2011    12
2012    12
2013    12
2014    12
2015     3
Name: time, dtype: int64


### Create Historical CNRM Seasonal Climate Features

Only complete historical CNRM hydrological years from 1968 through 2014 are retained. The data are then divided into the same winter and summer seasons used for the historical ERA5 data and future CNRM data. This creates comparable seasonal climate features that can be used to measure the difference between CNRM and ERA5.

In [21]:
historical_complete_years = (
    historical_months_per_year
    .where(historical_months_per_year == 12, drop=True)
    ["hydrological_year"]
    .values
)

historical_temperature_complete = historical_temperature_c.where(
    historical_temperature_c["hydrological_year"].isin(
        historical_complete_years
    ),
    drop=True
)

historical_precipitation_complete = historical_precipitation_mm.where(
    historical_precipitation_mm["hydrological_year"].isin(
        historical_complete_years
    ),
    drop=True
)

historical_winter_temperature = (
    historical_temperature_complete
    .where(
        historical_temperature_complete["time"].dt.month.isin(winter_months),
        drop=True
    )
    .groupby("hydrological_year")
    .mean(dim="time")
)

historical_summer_temperature = (
    historical_temperature_complete
    .where(
        historical_temperature_complete["time"].dt.month.isin(summer_months),
        drop=True
    )
    .groupby("hydrological_year")
    .mean(dim="time")
)

historical_winter_precipitation = (
    historical_precipitation_complete
    .where(
        historical_precipitation_complete["time"].dt.month.isin(winter_months),
        drop=True
    )
    .groupby("hydrological_year")
    .sum(dim="time")
)

historical_summer_precipitation = (
    historical_precipitation_complete
    .where(
        historical_precipitation_complete["time"].dt.month.isin(summer_months),
        drop=True
    )
    .groupby("hydrological_year")
    .sum(dim="time")
)

print(
    "Complete hydrological years:",
    int(historical_complete_years.min()),
    "to",
    int(historical_complete_years.max())
)

print("Number of complete years:", len(historical_complete_years))

print("\nWinter temperature shape:",
      historical_winter_temperature.shape)

print("Summer temperature shape:",
      historical_summer_temperature.shape)

print("Winter precipitation shape:",
      historical_winter_precipitation.shape)

print("Summer precipitation shape:",
      historical_summer_precipitation.shape)

Complete hydrological years: 1968 to 2014
Number of complete years: 47

Winter temperature shape: (47, 58)
Summer temperature shape: (47, 58)
Winter precipitation shape: (47, 58)
Summer precipitation shape: (47, 58)


### Combine Historical CNRM Seasonal Features

The historical CNRM seasonal variables are combined into one table with a row for each glacier and hydrological year. This gives the CNRM data the same basic structure as the ERA5 climate features in the historical model dataset so the two climate sources can be compared directly.

In [22]:
historical_cnrm_climate = pd.DataFrame({
    "hydrological_year": np.repeat(
        historical_winter_temperature["hydrological_year"].values,
        len(glacier_locations)
    ),
    "glacier_index": np.tile(
        glacier_locations["glacier_index"].values,
        len(historical_complete_years)
    ),
    "cnrm_winter_temperature_c":
        historical_winter_temperature.values.flatten(),
    "cnrm_summer_temperature_c":
        historical_summer_temperature.values.flatten(),
    "cnrm_winter_precipitation_mm":
        historical_winter_precipitation.values.flatten(),
    "cnrm_summer_precipitation_mm":
        historical_summer_precipitation.values.flatten()
})

print("Historical CNRM shape:", historical_cnrm_climate.shape)

print(
    "Hydrological years:",
    historical_cnrm_climate["hydrological_year"].min(),
    "to",
    historical_cnrm_climate["hydrological_year"].max()
)

print(
    "Glacier records:",
    historical_cnrm_climate["glacier_index"].nunique()
)

print(
    "Duplicate glacier-year rows:",
    historical_cnrm_climate.duplicated(
        subset=["glacier_index", "hydrological_year"]
    ).sum()
)

print("\nMissing values:")
print(historical_cnrm_climate.isnull().sum())

historical_cnrm_climate.head()

Historical CNRM shape: (2726, 6)
Hydrological years: 1968 to 2014
Glacier records: 58
Duplicate glacier-year rows: 0

Missing values:
hydrological_year               0
glacier_index                   0
cnrm_winter_temperature_c       0
cnrm_summer_temperature_c       0
cnrm_winter_precipitation_mm    0
cnrm_summer_precipitation_mm    0
dtype: int64


,hydrological_year,glacier_index,cnrm_winter_temperature_c,cnrm_summer_temperature_c,cnrm_winter_precipitation_mm,cnrm_summer_precipitation_mm
0,1968,423,-0.196509,10.429769,855.138436,1387.288164
1,1968,424,-0.182462,10.443675,854.985902,1382.717853
2,1968,427,-0.177229,10.450687,854.894737,1380.094338
3,1968,429,-0.236868,10.395934,855.383270,1397.488378
4,1968,431,-0.171988,10.456392,854.837337,1378.112512


### Compare Historical CNRM and ERA5 Climate

Historical CNRM climate features are matched with the ERA5 features for the same glacier records and hydrological years. The comparison is limited to their shared period from 1968 through 2014. This allows the differences between CNRM and the ERA5 climate data used during model development to be measured before correcting the future projections.

In [23]:
era5_features = historical_data[
    [
        "hydrological_year",
        "glacier_index",
        "winter_temperature_c",
        "summer_temperature_c",
        "winter_precipitation_mm",
        "summer_precipitation_mm"
    ]
].copy()

climate_comparison = era5_features.merge(
    historical_cnrm_climate,
    on=["hydrological_year", "glacier_index"],
    how="inner"
)

print("Comparison shape:", climate_comparison.shape)

print(
    "Hydrological years:",
    climate_comparison["hydrological_year"].min(),
    "to",
    climate_comparison["hydrological_year"].max()
)

print(
    "Glacier records:",
    climate_comparison["glacier_index"].nunique()
)

print(
    "Duplicate glacier-year rows:",
    climate_comparison.duplicated(
        subset=["glacier_index", "hydrological_year"]
    ).sum()
)

print("\nMissing values:")
print(climate_comparison.isnull().sum())

Comparison shape: (2671, 10)
Hydrological years: 1968 to 2014
Glacier records: 58
Duplicate glacier-year rows: 0

Missing values:
hydrological_year               0
glacier_index                   0
winter_temperature_c            0
summer_temperature_c            0
winter_precipitation_mm         0
summer_precipitation_mm         0
cnrm_winter_temperature_c       0
cnrm_summer_temperature_c       0
cnrm_winter_precipitation_mm    0
cnrm_summer_precipitation_mm    0
dtype: int64


### Measure CNRM Bias Relative to ERA5

The historical CNRM and ERA5 climate features are compared to estimate differences between the two datasets. Temperature bias is measured as the ERA5 value minus the CNRM value, while precipitation is compared using the ratio of ERA5 to CNRM. These values are inspected before any correction is applied to the future projections.

In [24]:
climate_comparison["winter_temperature_bias"] = (
    climate_comparison["winter_temperature_c"]
    - climate_comparison["cnrm_winter_temperature_c"]
)

climate_comparison["summer_temperature_bias"] = (
    climate_comparison["summer_temperature_c"]
    - climate_comparison["cnrm_summer_temperature_c"]
)

climate_comparison["winter_precipitation_ratio"] = (
    climate_comparison["winter_precipitation_mm"]
    / climate_comparison["cnrm_winter_precipitation_mm"]
)

climate_comparison["summer_precipitation_ratio"] = (
    climate_comparison["summer_precipitation_mm"]
    / climate_comparison["cnrm_summer_precipitation_mm"]
)

print("Average historical climate differences")

print("\nTemperature bias (ERA5 - CNRM)")
print(
    "Winter:",
    climate_comparison["winter_temperature_bias"].mean()
)
print(
    "Summer:",
    climate_comparison["summer_temperature_bias"].mean()
)

print("\nPrecipitation ratio (ERA5 / CNRM)")
print(
    "Winter:",
    climate_comparison["winter_precipitation_ratio"].mean()
)
print(
    "Summer:",
    climate_comparison["summer_precipitation_ratio"].mean()
)

Average historical climate differences

Temperature bias (ERA5 - CNRM)
Winter: -5.382631817309394
Summer: -3.9946475835268056

Precipitation ratio (ERA5 / CNRM)
Winter: 0.9301991341129281
Summer: 0.7341448522922283


### Check Bias Across Glacier Locations

The average comparison shows a clear difference between historical CNRM and ERA5 climate values. Because the future model inputs are created separately for each glacier location, the bias is also summarized by glacier to determine whether one correction can reasonably represent the entire study area or whether location specific corrections are more appropriate.

In [25]:
glacier_bias = (
    climate_comparison
    .groupby("glacier_index")
    .agg(
        winter_temperature_bias=("winter_temperature_bias", "mean"),
        summer_temperature_bias=("summer_temperature_bias", "mean"),
        winter_precipitation_ratio=("winter_precipitation_ratio", "mean"),
        summer_precipitation_ratio=("summer_precipitation_ratio", "mean")
    )
)

print("Bias across glacier locations\n")

for column in glacier_bias.columns:
    print(column)
    print("  Minimum:", glacier_bias[column].min())
    print("  Maximum:", glacier_bias[column].max())
    print("  Mean:", glacier_bias[column].mean())
    print("  Standard deviation:", glacier_bias[column].std())
    print()

Bias across glacier locations

winter_temperature_bias
  Minimum: -5.649901114627348
  Maximum: -5.154737454579353
  Mean: -5.380055150268674
  Standard deviation: 0.14508014133897518

summer_temperature_bias
  Minimum: -4.327466643201771
  Maximum: -3.7054365126367994
  Mean: -3.992533660363356
  Standard deviation: 0.17185607344247614

winter_precipitation_ratio
  Minimum: 0.8542392542539479
  Maximum: 1.0158066930013736
  Mean: 0.931771587865018
  Standard deviation: 0.04761848258571909

summer_precipitation_ratio
  Minimum: 0.667217778487566
  Maximum: 0.8344440907425118
  Mean: 0.7356525730526663
  Standard deviation: 0.04339610435319223



### Calculate Glacier-Specific Bias Corrections

The historical comparison shows that the CNRM bias varies somewhat across the Mont Blanc glacier locations. Glacier-specific corrections are therefore calculated so this spatial variation is preserved.

For temperature, the correction is the average ERA5 minus CNRM difference for each glacier and season. For precipitation, the correction is calculated as the ratio between the ERA5 and CNRM historical mean precipitation for each glacier and season. These corrections will be applied to the future CNRM climate features.

In [26]:
glacier_corrections = (
    climate_comparison
    .groupby("glacier_index")
    .agg(
        winter_temperature_bias=(
            "winter_temperature_bias", "mean"
        ),
        summer_temperature_bias=(
            "summer_temperature_bias", "mean"
        ),
        era5_winter_precipitation_mean=(
            "winter_precipitation_mm", "mean"
        ),
        cnrm_winter_precipitation_mean=(
            "cnrm_winter_precipitation_mm", "mean"
        ),
        era5_summer_precipitation_mean=(
            "summer_precipitation_mm", "mean"
        ),
        cnrm_summer_precipitation_mean=(
            "cnrm_summer_precipitation_mm", "mean"
        )
    )
    .reset_index()
)

glacier_corrections["winter_precipitation_ratio"] = (
    glacier_corrections["era5_winter_precipitation_mean"]
    / glacier_corrections["cnrm_winter_precipitation_mean"]
)

glacier_corrections["summer_precipitation_ratio"] = (
    glacier_corrections["era5_summer_precipitation_mean"]
    / glacier_corrections["cnrm_summer_precipitation_mean"]
)

glacier_corrections = glacier_corrections[
    [
        "glacier_index",
        "winter_temperature_bias",
        "summer_temperature_bias",
        "winter_precipitation_ratio",
        "summer_precipitation_ratio"
    ]
]

print("Glacier corrections shape:", glacier_corrections.shape)

print("\nCorrection ranges:")
for column in glacier_corrections.columns[1:]:
    print(
        column,
        ":",
        glacier_corrections[column].min(),
        "to",
        glacier_corrections[column].max()
    )

print("\nMissing values:")
print(glacier_corrections.isnull().sum())

glacier_corrections.head()

Glacier corrections shape: (58, 5)

Correction ranges:
winter_temperature_bias : -5.649901114627348 to -5.154737454579353
summer_temperature_bias : -4.327466643201771 to -3.7054365126367994
winter_precipitation_ratio : 0.8398831137732504 to 0.9975633143508688
summer_precipitation_ratio : 0.6588663076763887 to 0.8213338585109282

Missing values:
glacier_index                 0
winter_temperature_bias       0
summer_temperature_bias       0
winter_precipitation_ratio    0
summer_precipitation_ratio    0
dtype: int64


,glacier_index,winter_temperature_bias,summer_temperature_bias,winter_precipitation_ratio,summer_precipitation_ratio
0,423,-5.227823,-3.803133,0.969637,0.699427
1,424,-5.215613,-3.788982,0.971319,0.707510
2,427,-5.214185,-3.788415,0.970334,0.712196
3,429,-5.272232,-3.858857,0.959734,0.681876
4,431,-5.210637,-3.784538,0.970469,0.715697


### Apply Bias Corrections to Future Climate Data

The glacier-specific historical corrections are applied to the future CNRM climate features. Temperature is corrected by adding the historical ERA5 minus CNRM bias, while precipitation is corrected by multiplying the future CNRM values by the historical ERA5 to CNRM precipitation ratio.

This adjusts the future CNRM projections to the climate scale used during model training while preserving the future changes projected by CNRM-CM6-1.

In [27]:
future_climate_corrected = future_climate.merge(
    glacier_corrections,
    on="glacier_index",
    how="left"
)

future_climate_corrected["winter_temperature_c"] = (
    future_climate_corrected["winter_temperature_c"]
    + future_climate_corrected["winter_temperature_bias"]
)

future_climate_corrected["summer_temperature_c"] = (
    future_climate_corrected["summer_temperature_c"]
    + future_climate_corrected["summer_temperature_bias"]
)

future_climate_corrected["winter_precipitation_mm"] = (
    future_climate_corrected["winter_precipitation_mm"]
    * future_climate_corrected["winter_precipitation_ratio"]
)

future_climate_corrected["summer_precipitation_mm"] = (
    future_climate_corrected["summer_precipitation_mm"]
    * future_climate_corrected["summer_precipitation_ratio"]
)

print("Corrected future climate shape:",
      future_climate_corrected.shape)

print(
    "Hydrological years:",
    future_climate_corrected["hydrological_year"].min(),
    "to",
    future_climate_corrected["hydrological_year"].max()
)

print(
    "Glacier records:",
    future_climate_corrected["glacier_index"].nunique()
)

print("\nCorrected climate ranges:")

for column in [
    "winter_temperature_c",
    "summer_temperature_c",
    "winter_precipitation_mm",
    "summer_precipitation_mm"
]:
    print(
        column,
        ":",
        future_climate_corrected[column].min(),
        "to",
        future_climate_corrected[column].max()
    )

print("\nMissing values:")
print(future_climate_corrected.isnull().sum())

Corrected future climate shape: (2900, 12)
Hydrological years: 2027 to 2076
Glacier records: 58

Corrected climate ranges:
winter_temperature_c : -5.670051394155838 to -0.721812353348251
summer_temperature_c : 6.912093196191347 to 10.289229211316112
winter_precipitation_mm : 562.5780665308034 to 1267.7396188071295
summer_precipitation_mm : 569.6850692335723 to 1224.6754638891919

Missing values:
hydrological_year             0
glacier_index                 0
winter_temperature_c          0
summer_temperature_c          0
winter_precipitation_mm       0
summer_precipitation_mm       0
longitude                     0
latitude                      0
winter_temperature_bias       0
summer_temperature_bias       0
winter_precipitation_ratio    0
summer_precipitation_ratio    0
dtype: int64


### Create Final Future Climate Dataset

The bias correction columns are removed after the corrections are applied. The remaining dataset contains the glacier location information and the four seasonal climate features required by the final Linear Regression model. The dataset represents 50 complete future hydrological years from 2027 through 2076 under the CNRM-CM6-1 SSP2-4.5 scenario.

In [28]:
future_model_data = future_climate_corrected[
    [
        "hydrological_year",
        "glacier_index",
        "longitude",
        "latitude",
        "winter_temperature_c",
        "summer_temperature_c",
        "winter_precipitation_mm",
        "summer_precipitation_mm"
    ]
].copy()

print("Final future dataset shape:", future_model_data.shape)

print(
    "Hydrological years:",
    future_model_data["hydrological_year"].min(),
    "to",
    future_model_data["hydrological_year"].max()
)

print(
    "Glacier records:",
    future_model_data["glacier_index"].nunique()
)

print(
    "Duplicate glacier-year rows:",
    future_model_data.duplicated(
        subset=["glacier_index", "hydrological_year"]
    ).sum()
)

print("\nMissing values:")
print(future_model_data.isnull().sum())

future_model_data.head()

Final future dataset shape: (2900, 8)
Hydrological years: 2027 to 2076
Glacier records: 58
Duplicate glacier-year rows: 0

Missing values:
hydrological_year          0
glacier_index              0
longitude                  0
latitude                   0
winter_temperature_c       0
summer_temperature_c       0
winter_precipitation_mm    0
summer_precipitation_mm    0
dtype: int64


,hydrological_year,glacier_index,longitude,latitude,winter_temperature_c,summer_temperature_c,winter_precipitation_mm,summer_precipitation_mm
0,2027,423,6.776,45.799,-2.543615,8.365329,880.134213,742.061964
1,2027,424,6.778,45.809,-2.520858,8.392467,882.112250,749.987557
2,2027,427,6.783,45.817,-2.515188,8.399441,881.820691,754.815835
3,2027,429,6.784,45.784,-2.617241,8.277567,871.183794,725.533735
4,2027,431,6.785,45.822,-2.507621,8.408603,882.249182,758.315142


### Compare Historical and Future Climate Ranges

The corrected future climate features are compared with the historical ERA5 features used for model development. Future values are not expected to remain completely within the historical range because the climate projections represent changing future conditions. This comparison is used as a final check for unrealistic values or possible processing errors before the future dataset is saved.

In [29]:
feature_columns = [
    "winter_temperature_c",
    "summer_temperature_c",
    "winter_precipitation_mm",
    "summer_precipitation_mm"
]

comparison_rows = []

for feature in feature_columns:
    comparison_rows.append({
        "feature": feature,
        "historical_min": historical_data[feature].min(),
        "historical_max": historical_data[feature].max(),
        "future_min": future_model_data[feature].min(),
        "future_max": future_model_data[feature].max()
    })

range_comparison = pd.DataFrame(comparison_rows)

range_comparison

,feature,historical_min,historical_max,future_min,future_max
0,winter_temperature_c,-6.618042,-1.480644,-5.670051,-0.721812
1,summer_temperature_c,4.880237,9.848639,6.912093,10.289229
2,winter_precipitation_mm,432.036533,1305.799643,562.578067,1267.739619
3,summer_precipitation_mm,560.802422,1077.655710,569.685069,1224.675464


### Save Future Climate Data

The final bias-corrected future climate dataset is saved for use in the glacier mass balance forecasting stage. It contains the four climate features required by the final Linear Regression model for all 58 glacier records across 50 complete hydrological years from 2027 through 2076.

In [30]:
future_output_file = (
    PROCESSED_DIR /
    "MontBlanc_Future_Climate_SSP245_2027-2076.csv"
)

future_model_data.to_csv(
    future_output_file,
    index=False
)

print("Saved:", future_output_file)
print("Shape:", future_model_data.shape)
print(
    "Years:",
    future_model_data["hydrological_year"].min(),
    "to",
    future_model_data["hydrological_year"].max()
)
print(
    "Glacier records:",
    future_model_data["glacier_index"].nunique()
)

Saved: ../data/processed/MontBlanc_Future_Climate_SSP245_2027-2076.csv
Shape: (2900, 8)
Years: 2027 to 2076
Glacier records: 58


### Future Climate Preparation Summary

Future CNRM-CM6-1 SSP2-4.5 temperature and precipitation projections were prepared for the 58 Mont Blanc glacier records from hydrological years 2027 through 2076. The climate data were converted to the same units, seasons, and glacier locations used by the historical ERA5 model data.

Historical CNRM data were compared with ERA5 over their shared historical period to calculate glacier-specific seasonal bias corrections. Temperature was corrected using the historical ERA5 minus CNRM difference, while precipitation was corrected using the ratio of historical ERA5 to CNRM precipitation.

The final dataset contains 2,900 glacier-year observations with the four climate features required by the selected Linear Regression model. No missing values or duplicate glacier-year records remain. The processed future climate data are now ready for future glacier mass balance prediction.